In [2]:
import pandas as pd
import numpy as np

file_path = "/Users/shusilkhulal/Downloads/online_retail_II.xlsx"

df = pd.read_excel(file_path)

df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [4]:
# data cleaning
df.isnull().sum()

Invoice             0
StockCode           0
Description      2928
Quantity            0
InvoiceDate         0
Price               0
Customer ID    107927
Country             0
dtype: int64

In [5]:
df = df.dropna(subset=["Customer ID"])

In [6]:
df = df[df["Quantity"] > 0]

In [7]:
df = df[df["Price"] > 0]

In [8]:
df["Revenue"] = df["Quantity"] * df["Price"]

In [10]:
# exploratory analysis
print("Total Transactions:", len(df))

Total Transactions: 407664


In [11]:
print("Unique Customers:", df["Customer ID"].nunique())

Unique Customers: 4312


In [12]:
print("Start Date:", df["InvoiceDate"].min())
print("End Date:", df["InvoiceDate"].max())

Start Date: 2009-12-01 07:45:00
End Date: 2010-12-09 20:01:00


In [13]:
print("Total Revenue: ${:,.2f}".format(df["Revenue"].sum()))

Total Revenue: $8,832,003.27


In [14]:
print("Average Order Value: ${:,.2f}".format(df["Revenue"].mean()))

Average Order Value: $21.66


In [15]:
country_summary = (
    df.groupby("Country")["Revenue"]
      .sum()
      .sort_values(ascending=False)
      .head(10)
)

print(country_summary)

Country
United Kingdom    7414755.963
EIRE               356085.210
Netherlands        268786.000
Germany            202395.321
France             146215.420
Sweden              53171.390
Denmark             50906.850
Spain               47601.420
Switzerland         43921.390
Australia           31446.800
Name: Revenue, dtype: float64


In [16]:
#RFM
snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

In [18]:
rfm = df.groupby("Customer ID").agg(
    Recency=("InvoiceDate", lambda x: (snapshot_date - x.max()).days),
    Frequency=("Invoice", "nunique"),
    Monetary=("Revenue", "sum")
).reset_index()
rfm.head()

,Customer ID,Recency,Frequency,Monetary
0,12346.0,165,11,372.86
1,12347.0,3,2,1323.32
2,12348.0,74,1,222.16
3,12349.0,43,3,2671.14
4,12351.0,11,1,300.93


In [20]:
#RFM scoring
# Recency Score
rfm["R_Score"] = pd.qcut(
    rfm["Recency"].rank(method="first"),
    5,
    labels=[5,4,3,2,1]
)

# Frequency Score
rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    5,
    labels=[1,2,3,4,5]
)

# Monetary Score
rfm["M_Score"] = pd.qcut(
    rfm["Monetary"].rank(method="first"),
    5,
    labels=[1,2,3,4,5]
)

In [23]:
rfm["RFM_Score"] = (
    rfm["R_Score"].astype(str) +
    rfm["F_Score"].astype(str) +
    rfm["M_Score"].astype(str)
)
rfm.head()

,Customer ID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score
0,12346.0,165,11,372.86,2,5,2,252
1,12347.0,3,2,1323.32,5,2,4,524
2,12348.0,74,1,222.16,2,1,1,211
3,12349.0,43,3,2671.14,3,3,5,335
4,12351.0,11,1,300.93,5,1,2,512


In [24]:
#customer segementation
def segment_customer(row):

    if row["R_Score"] >= 4 and row["F_Score"] >= 4:
        return "Champions"

    elif row["R_Score"] >= 3 and row["F_Score"] >= 2:
        return "Potential Loyalists"

    elif row["R_Score"] >= 2 and row["F_Score"] >= 3:
        return "Loyal Customers"

    elif row["R_Score"] >= 4 and row["F_Score"] == 1:
        return "New / Recent Customers"

    elif row["R_Score"] <= 2 and row["F_Score"] >= 4:
        return "At-Risk High Value"

    else:
        return "Lost Customers"

In [26]:
rfm["Segment"] = rfm.apply(segment_customer, axis=1)
rfm.head()

,Customer ID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment
0,12346.0,165,11,372.86,2,5,2,252,Loyal Customers
1,12347.0,3,2,1323.32,5,2,4,524,Potential Loyalists
2,12348.0,74,1,222.16,2,1,1,211,Lost Customers
3,12349.0,43,3,2671.14,3,3,5,335,Potential Loyalists
4,12351.0,11,1,300.93,5,1,2,512,New / Recent Customers


In [27]:
segment_summary = (
    rfm["Segment"]
    .value_counts()
    .reset_index()
)

segment_summary.columns = ["Segment", "Customers"]

segment_summary

,Segment,Customers
0,Lost Customers,1346
1,Potential Loyalists,1193
2,Champions,1084
3,Loyal Customers,483
4,New / Recent Customers,132
5,At-Risk High Value,74


In [28]:
#segment summary
segment_summary = (
    rfm.groupby("Segment")
       .agg(
            Customers=("Customer ID","count"),
            Avg_Frequency=("Frequency","mean"),
            Avg_Recency=("Recency","mean"),
            Revenue=("Monetary","sum")
       )
       .reset_index()
)

segment_summary

,Segment,Customers,Avg_Frequency,Avg_Recency,Revenue
0,At-Risk High Value,74,4.810811,230.270270,162171.710
1,Champions,1084,10.524908,13.697417,5786240.987
2,Lost Customers,1346,1.230312,194.089896,612148.224
3,Loyal Customers,483,4.074534,108.134576,781079.991
4,New / Recent Customers,132,1.000000,18.856061,46875.790
5,Potential Loyalists,1193,3.094719,37.956412,1443486.572


In [29]:
segment_summary["Avg_Frequency"] = segment_summary["Avg_Frequency"].round(1)
segment_summary["Avg_Recency"] = segment_summary["Avg_Recency"].round(0)
segment_summary["Revenue"] = segment_summary["Revenue"].round(2)

segment_summary

,Segment,Customers,Avg_Frequency,Avg_Recency,Revenue
0,At-Risk High Value,74,4.8,230.0,162171.71
1,Champions,1084,10.5,14.0,5786240.99
2,Lost Customers,1346,1.2,194.0,612148.22
3,Loyal Customers,483,4.1,108.0,781079.99
4,New / Recent Customers,132,1.0,19.0,46875.79
5,Potential Loyalists,1193,3.1,38.0,1443486.57


In [30]:
# customer life time value
rfm["CLV"] = rfm["Monetary"]

In [32]:
rfm["CLV_Segment"] = pd.qcut(
    rfm["CLV"],
    q=4,
    labels=[
        "Low Value",
        "Medium Value",
        "High Value",
        "VIP Customers"
    ]
)
rfm.head()

,Customer ID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment,CLV,CLV_Segment
0,12346.0,165,11,372.86,2,5,2,252,Loyal Customers,372.86,Medium Value
1,12347.0,3,2,1323.32,5,2,4,524,Potential Loyalists,1323.32,High Value
2,12348.0,74,1,222.16,2,1,1,211,Lost Customers,222.16,Low Value
3,12349.0,43,3,2671.14,3,3,5,335,Potential Loyalists,2671.14,VIP Customers
4,12351.0,11,1,300.93,5,1,2,512,New / Recent Customers,300.93,Low Value


In [33]:
#clv summary
clv_summary = (
    rfm.groupby("CLV_Segment")
       .agg(
            Customers=("Customer ID","count"),
            Revenue=("CLV","sum"),
            Avg_CLV=("CLV","mean")
       )
       .reset_index()
)

clv_summary

/var/folders/49/2t3w2v415lz6r6c3r4sbnk9w0000gn/T/ipykernel_12947/1135747455.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  rfm.groupby("CLV_Segment")


,CLV_Segment,Customers,Revenue,Avg_CLV
0,Low Value,1078,192573.080,178.639221
1,Medium Value,1078,513414.153,476.265448
2,High Value,1078,1220622.590,1132.302959
3,VIP Customers,1078,6905393.451,6405.745316


In [34]:
#customer retention and cohort analysis
df["InvoiceMonth"] = df["InvoiceDate"].dt.to_period("M")

In [35]:
cohort = (
    df.groupby("Customer ID")["InvoiceMonth"]
      .min()
      .rename("CohortMonth")
)

df = df.merge(cohort, on="Customer ID")

In [36]:
df[["Customer ID","InvoiceMonth","CohortMonth"]].head()

,Customer ID,InvoiceMonth,CohortMonth
0,13085.0,2009-12,2009-12
1,13085.0,2009-12,2009-12
2,13085.0,2009-12,2009-12
3,13085.0,2009-12,2009-12
4,13085.0,2009-12,2009-12


In [37]:
#chort index
invoice_year = df["InvoiceMonth"].dt.year
invoice_month = df["InvoiceMonth"].dt.month

cohort_year = df["CohortMonth"].dt.year
cohort_month = df["CohortMonth"].dt.month

df["CohortIndex"] = (
    (invoice_year - cohort_year) * 12
    + (invoice_month - cohort_month)
    + 1
)

In [39]:
# active customer
cohort_data = (
    df.groupby(["CohortMonth","CohortIndex"])
      .agg(Customers=("Customer ID","nunique"))
      .reset_index()
)

cohort_data.head()

,CohortMonth,CohortIndex,Customers
0,2009-12,1,955
1,2009-12,2,337
2,2009-12,3,319
3,2009-12,4,406
4,2009-12,5,363


In [41]:
# cohort size
cohort_size = (
    cohort_data[cohort_data["CohortIndex"] == 1]
    [["CohortMonth","Customers"]]
    .rename(columns={"Customers":"CohortSize"})
)

cohort_data = cohort_data.merge(
    cohort_size,
    on="CohortMonth"
)

In [43]:
# retention rate
cohort_data["RetentionRate"] = (
    cohort_data["Customers"] /
    cohort_data["CohortSize"]
)

In [44]:
cohort_data["RetentionRate"] = (
    cohort_data["RetentionRate"] * 100
).round(1)

In [45]:
# heat map table
cohort_retention = cohort_data.pivot(
    index="CohortMonth",
    columns="CohortIndex",
    values="RetentionRate"
)

cohort_retention


CohortIndex,1,2,3,4,5,6,7,8,9,10,11,12,13
CohortMonth,,,,,,,,,,,,,
2009-12,100.0,35.3,33.4,42.5,38.0,35.9,37.7,34.2,33.6,36.2,42.2,49.5,24.8
2010-01,100.0,20.6,31.1,30.5,26.4,30.0,25.8,23.0,27.9,31.9,30.3,9.9,NaN
2010-02,100.0,23.8,22.5,29.1,24.6,20.1,19.3,28.6,25.4,27.5,7.2,NaN,NaN
2010-03,100.0,19.0,23.0,24.2,23.3,20.3,24.6,30.2,27.5,7.9,NaN,NaN,NaN
2010-04,100.0,19.4,19.4,16.3,18.4,22.4,27.6,26.2,6.8,NaN,NaN,NaN,NaN
2010-05,100.0,15.7,16.9,17.3,17.7,25.6,21.3,7.9,NaN,NaN,NaN,NaN,NaN
2010-06,100.0,17.4,18.9,20.4,23.0,28.5,6.7,NaN,NaN,NaN,NaN,NaN,NaN
2010-07,100.0,15.6,18.3,29.6,29.0,10.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-08,100.0,20.4,29.6,32.1,11.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [46]:
cohort_retention_long = cohort_data[
    ["CohortMonth","CohortIndex","RetentionRate"]
]

cohort_retention_long.head()

,CohortMonth,CohortIndex,RetentionRate
0,2009-12,1,100.0
1,2009-12,2,35.3
2,2009-12,3,33.4
3,2009-12,4,42.5
4,2009-12,5,38.0


In [47]:
# average retention month 1
avg_retention = cohort_data.loc[
    cohort_data["CohortIndex"] > 1,
    "RetentionRate"
].mean()

print(f"Average Retention Rate: {avg_retention:.1f}%")

Average Retention Rate: 23.7%


In [48]:
# average retention month 2
month2_retention = cohort_data.loc[
    cohort_data["CohortIndex"] == 2,
    "RetentionRate"
].mean()

print(f"Average Month 2 Retention: {month2_retention:.1f}%")

Average Month 2 Retention: 20.5%


In [49]:
# average retention month 3
month3_retention = cohort_data.loc[
    cohort_data["CohortIndex"] == 3,
    "RetentionRate"
].mean()

print(f"Average Month 3 Retention: {month3_retention:.1f}%")

Average Month 3 Retention: 22.4%


In [51]:
rfm.to_csv("customer_value_dashboard.csv", index=False)
cohort_retention_long.to_csv("cohort_retention_long.csv", index=False)